# Goal 3 — PageValues ablation (deployment leakage)

Week 1 I kept `PageValues` and treated the model as more of an end-of-session / offline scorer. This notebook turns that into a proper experiment:

1. Train the **same** CatBoost setup twice (same split, same hyperparameters)
2. Once **with** `PageValues`, once **without**
3. Compare PR-AUC / precision / recall
4. Read the gap as a **deployment-leakage** issue, not just “a useful feature”


In [1]:
import sys
from pathlib import Path

ROOT = Path(r"C:\Users\saksh\OneDrive - University of Keele\Desktop\Ecommerce_project")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sklearn.model_selection import train_test_split

from src.config.settings import load_config
from src.data.loader import load_data
from src.evaluation.ablation import run_pagevalues_ablation
from src.features.engineering import add_engineered_features


In [2]:
config = load_config()
df = add_engineered_features(load_data())
X = df.drop(columns=[config["target_column"]])
y = df[config["target_column"]]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=config["test_size"],
    random_state=config["random_seed"],
)

table = run_pagevalues_ablation(
    X_train,
    X_test,
    y_train,
    y_test,
    categorical_columns=config["categorical_columns"],
    threshold=float(config["threshold"]),
)

out = ROOT / "reports" / "pagevalues_ablation.csv"
out.parent.mkdir(exist_ok=True)
table.to_csv(out, index=False)
print(f"Saved: {out}")
table


2026-09-02 19:45:56,113 | INFO | src.data.loader | Data loaded successfully from C:\Users\saksh\OneDrive - University of Keele\Desktop\Ecommerce_project\dataset\ecommerce_sessions.csv.
2026-09-02 19:45:56,114 | INFO | src.data.loader | All expected columns are present in the dataset.
2026-09-02 19:45:56,142 | INFO | src.features.engineering | Engineered features added to the DataFrame.


Saved: C:\Users\saksh\OneDrive - University of Keele\Desktop\Ecommerce_project\reports\pagevalues_ablation.csv


,variant,n_features_raw,PR-AUC,Precision,Recall,F1
0,with_PageValues,21,0.8628,0.8177,0.7653,0.7906
1,without_PageValues,20,0.5545,0.6316,0.3840,0.4776


In [3]:
with_row = table.loc[table["variant"] == "with_PageValues"].iloc[0]
without_row = table.loc[table["variant"] == "without_PageValues"].iloc[0]

print("Drop when removing PageValues:")
for metric in ["PR-AUC", "Precision", "Recall", "F1"]:
    delta = without_row[metric] - with_row[metric]
    print(f"  {metric}: {with_row[metric]:.4f} -> {without_row[metric]:.4f}  ({delta:+.4f})")


Drop when removing PageValues:
  PR-AUC: 0.8628 -> 0.5545  (-0.3083)
  Precision: 0.8177 -> 0.6316  (-0.1861)
  Recall: 0.7653 -> 0.3840  (-0.3813)
  F1: 0.7906 -> 0.4776  (-0.3130)


## Metrics table (same test split)

| Variant | PR-AUC | Precision @ 0.5 | Recall @ 0.5 | F1 |
|---|---:|---:|---:|---:|
| With `PageValues` | ~0.86 | ~0.82 | ~0.77 | ~0.79 |
| Without `PageValues` | ~0.55 | ~0.63 | ~0.38 | ~0.48 |

(Exact numbers come from the cell above / `reports/pagevalues_ablation.csv`.)

## Deployment-leakage read

This isn’t just “removing a strong feature made the model worse.” `PageValues` is tied to pages close to checkout, so for a **live early-session** score you often wouldn’t have a meaningful value yet. In that setting, the **without PageValues** numbers (~0.55 PR-AUC, much lower recall) are probably the more honest estimate of what you’d get in production.

I’m still keeping `PageValues` in the main project model because I framed the use case as **end-of-session / offline** scoring in Week 1. If the product requirement changed to “score them in the first 30 seconds,” I’d ship the no-`PageValues` model (or train a second early-session model) and accept the weaker metrics rather than pretend the 0.86 PR-AUC still applies.
